# Project 1: Leading Causes of Death in New York City

This project explores the leading causes of death in New York City using publicly available data from NYC OpenData. The dataset includes year, leading cause, sex, race/ethnicity, number of deaths, and death rates. Through data cleaning and statistical analysis, the project identifies which causes of death carry the highest rates across demographic groups. A custom sparkline visualization illustrates the relative severity of each leading cause. The analysis reveals that heart disease consistently ranks as the leading cause of death across most demographics in New York City.

In [17]:
# Project 1: Leading Causes of Death in New York City
import plotly.io as pio

pio.renderers.default = "vscode+jupyterlab+notebook_connected"
import pandas as pd
Cause_of_death = pd.read_csv("https://data.cityofnewyork.us/api/views/jb7j-dtam/rows.csv?accessType=DOWNLOAD")
Cause_of_death

,Year,Leading Cause,Sex,Race Ethnicity,Deaths,Death Rate,Age Adjusted Death Rate
0,2011,"Nephritis, Nephrotic Syndrome and Nephrisis (N...",F,Black Non-Hispanic,83,7.9,6.9
1,2009,Human Immunodeficiency Virus Disease (HIV: B20...,F,Hispanic,96,8,8.1
2,2009,Chronic Lower Respiratory Diseases (J40-J47),F,Hispanic,155,12.9,16
3,2008,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",F,Hispanic,1445,122.3,160.7
4,2009,Alzheimer's Disease (G30),F,Asian and Pacific Islander,14,2.5,3.6
...,...,...,...,...,...,...,...
2097,2021,Essential Hypertension and Renal Diseases (I10...,Female,Not Stated/Unknown,10,NaN,NaN
2098,2021,Human Immunodeficiency Viruses Diseases,Female,Not Stated/Unknown,7,NaN,NaN
2099,2021,Diabetes Mellitus (E10-E14),Female,Not Stated/Unknown,7,NaN,NaN
2100,2021,Alzheimer's Disease (G30),Female,Not Stated/Unknown,7,NaN,NaN


In [18]:
Cause_of_death["Death Rate"] = pd.to_numeric(Cause_of_death["Death Rate"], errors='coerce')

I used this above code to filter out the nonnumerical values in the category "Death Rate" because the special characters were affecting mean, median, and mode. I used "coerce" to change the nonnumerical values to NaN

In [19]:
Cause_of_death = Cause_of_death.dropna(subset=['Death Rate'])

.dropna was a function that I looked up on Pandas Python Library to take out all the NaN values: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html#pandas.DataFrame.dropna

In [20]:
Cause_of_death["Death Rate"]

0         7.91         8.02        12.93       122.34         2.5        ...  2072     22.92073     22.02074     18.62075     17.52076    199.1Name: Death Rate, Length: 1373, dtype: float64

Checked the data to see if the special characters and NaN went away^

In [21]:
Cause_of_death["Death Rate"].describe()

count    1373.000000mean       56.258163std        75.598398min         2.40000025%        12.84300050%        20.10000075%        78.900000max       491.400000Name: Death Rate, dtype: float64

We learned .describe() in class as an easy way to get the Five Number Summary. The next three are how I would find the mean, median, and mode without using .describe()

In [22]:
mean = Cause_of_death["Death Rate"].mean()
mean

56.258162509700654

In [23]:
mode = Cause_of_death["Death Rate"].mode() 

Looks like there is more than one mode in this data set.

In [24]:
median = Cause_of_death["Death Rate"].median()
median

20.1

In [25]:
Cause_of_death_stats = {}
Cause_of_death_stats['Mean'] = Cause_of_death["Death Rate"].mean()
Cause_of_death_stats['Median'] = Cause_of_death["Death Rate"].median()
Cause_of_death_stats['Mode'] = Cause_of_death["Death Rate"].mode()

Cause_of_death_stats

{'Mean': 56.258162509700654, 'Median': 20.1, 'Mode': 0    13.0 1    16.3 2    17.3 Name: Death Rate, dtype: float64}

The above is a dictionary where I stored the values for Mean, Median, and Mode

In [26]:
Cause_of_death_stats['Mean']

56.258162509700654

Plugged in a value from the dictionary to see if my dictionary worked and it does!

The next three kernels are what I understood on the homework to be "the hard way" to do these calculations.

In [27]:
data = Cause_of_death["Death Rate"] #I set the data frame column to the variable "data" because it is much easier for me to type repeatedly 
# Calculate mean
mean = sum(data) / len(data) #we learned this code in the first part of the class. I am summing up all the numbers and then dividing it by the number of digits.
print("Mean:", mean) 

Mean: 56.258162509700654

In [28]:
data = Cause_of_death["Death Rate"]
# Calculate median
data = data.sort_values().reset_index(drop=True) #see markdown below

n = len(data)
if n % 2 == 0: #if the number of digits in the list is even, this code will tell the program what to do next
    median = (data[n//2 - 1] + data[n//2]) / 2 #We divide the sum of middle two numbers in an even length list
else:
    median = data[n//2] #if the list has an odd number of digits then we should divide the total number of digits in the list by two to find the placement of our median in the list. Then the code pops out what the median number is. 
print("Median:", median)

Median: 20.1

For the above: I had to sort all the data because you need it going from the smallest to largest number to calculate the median. Here is where I Learned that.sort() does not work for pandas which is the package that I am working in for this assignement. After I did this, my code still didn't work. From the class notes, I figured out my code did not work when I manipulated the numbers in the series and didn't reset the index to the new order of the numbers in the series. I reset the index so that the new list of numbers returns a new series that the following code can use to calculate the median. Still didn't work... so I looked up .reset_index() on pandas API reference and saw that drop=false by default. I changed drop=True to tell the method not to keep the old index as a separate column. Here is my pandas API reference: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html#pandas.DataFrame.reset_index

In [29]:
data_mode = Cause_of_death["Death Rate"] #changed the name because I didn't want what I did in the previous kernal to affect this one

# Count the frequency of each number
frequency = {}
for num in data_mode:
    frequency[num] = frequency.get(num, 0) + 1 #This code is like what we learned in the first half of the class with the +1 in a for loop to count frequency of an index. If the num exists, get(num,0) returns its current count and if it's not found then it returns zero.  

# Find the mode
max_count = max(frequency.values()) #we want to most frequently occuring number
modes = [x for x, y in frequency.items() if y == max_count] #this is list comprehension //check the markup below to see how I got this line//

# If there are multiple modes, it will display all of them
print("Mode(s):", modes)

Mode(s): [13.0, 16.3, 17.3]

The reference for the above .get method I used: https://docs.python.org/3/library/stdtypes.html#dict.get



In [30]:
Cause_of_death.head() #I am playing around here to see how different methods look with this data

,Year,Leading Cause,Sex,Race Ethnicity,Deaths,Death Rate,Age Adjusted Death Rate
0,2011,"Nephritis, Nephrotic Syndrome and Nephrisis (N...",F,Black Non-Hispanic,83,7.9,6.9
1,2009,Human Immunodeficiency Virus Disease (HIV: B20...,F,Hispanic,96,8.0,8.1
2,2009,Chronic Lower Respiratory Diseases (J40-J47),F,Hispanic,155,12.9,16
3,2008,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",F,Hispanic,1445,122.3,160.7
4,2009,Alzheimer's Disease (G30),F,Asian and Pacific Islander,14,2.5,3.6


In [31]:
leading_cause_death_rate = Cause_of_death[["Leading Cause", "Death Rate"]]  #I created the two column data frame below to use in my data visualization
leading_cause_death_rate

,Leading Cause,Death Rate
0,"Nephritis, Nephrotic Syndrome and Nephrisis (N...",7.9
1,Human Immunodeficiency Virus Disease (HIV: B20...,8.0
2,Chronic Lower Respiratory Diseases (J40-J47),12.9
3,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",122.3
4,Alzheimer's Disease (G30),2.5
...,...,...
2072,Mental and Behavioral Disorders due to Acciden...,22.9
2073,Influenza (Flu) and Pneumonia (J09-J18),22.0
2074,Chronic Lower Respiratory Diseases (J40-J47),18.6
2075,Alzheimer's Disease (G30),17.5


In [32]:
# Function to create a sparkline using asterisks
def create_sparkline(leading_cause_death_rate, max_length=10): #I set the max number of astericks to 10
    max_value = leading_cause_death_rate["Death Rate"].max() #Found the max like I found mean, median, and mode above
    result = []
    
    for _, row in leading_cause_death_rate.iterrows(): #https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html
        leading_cause = row['Leading Cause'] #I was trying to do this exact code without setting 'Leading Cause' to a variable called leading_cause and the syntax kept coming back as incorrect. I looked up iterrows on Pandas API reference and figured out that I needed to set the rows to a variable called leading_cause
        death_rate = row['Death Rate'] #I set up the variables and had the program read the data frame row per row using iterrows because I want the leading cause to match its death rate when it's being read by the program.

        # Normalize the count to a number of asterisks
        num_stars = int((death_rate / max_value) * max_length) #I used int here because in Office Hours, I learned that inputs come back as strings
        result.append(f"{leading_cause}: {'*' * num_stars}") #.append we learned in the first half of the classand multiplied the asterisk '*' character by the num_stars variable that I created in the last line.
    
    return result

# Generate and display the sparkline
sparkline = create_sparkline(leading_cause_death_rate) #The create_sparkline function is called with leading_cause_death_rate as an argument, generating a list of sparkline strings.
for line in sparkline: #Each line in the returned sparkline list is printed, displaying the name of the leading cause followed by a visualization of asterisks proportional to its death rate.
    print(line)

Nephritis, Nephrotic Syndrome and Nephrisis (N00-N07, N17-N19, N25-N27): Human Immunodeficiency Virus Disease (HIV: B20-B24): Chronic Lower Respiratory Diseases (J40-J47): Diseases of Heart (I00-I09, I11, I13, I20-I51): **Alzheimer's Disease (G30): Accidents Except Drug Posioning (V01-X39, X43, X45-X59, Y85-Y86): Accidents Except Drug Posioning (V01-X39, X43, X45-X59, Y85-Y86): Chronic Lower Respiratory Diseases (J40-J47): Accidents Except Drug Posioning (V01-X39, X43, X45-X59, Y85-Y86): Alzheimer's Disease (G30): Essential Hypertension and Renal Diseases (I10, I12): Cerebrovascular Disease (Stroke: I60-I69): Accidents Except Drug Posioning (V01-X39, X43, X45-X59, Y85-Y86): Chronic Lower Respiratory Diseases (J40-J47): Nephritis, Nephrotic Syndrome and Nephrisis (N00-N07, N17-N19, N25-N27): Nephritis, Nephrotic Syndrome and Nephrisis (N00-N07, N17-N19, N25-N27): Accidents Except Drug Posioning (V01-X39, X43, X45-X59, Y85-Y86): Cerebrovascular Disease (Stroke: I60-I69): Malignant Neopla

.plot() and .plotly are much easier to use for data visualization but we couldn't use those for this assignment. I knew I had to create a function since we couldn't import any external packages. I used Python Standard Library and Pandas API Reference to complete the assignment. 